IMPORT e set

In [1]:
import os
import configparser
import torch 
import numpy as np 
from torchvision import transforms
from torch.utils.data import DataLoader
from ut import BalanceSet
from PIL import Image
import os
import configparser
import torch 
import numpy as np 
from torchvision import transforms
from torch.utils.data import DataLoader
from model import Network
from ut import GiveMeSampleSize, BalanceSet
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import plotly.io as pio
import configparser
from torcheval.metrics.functional import multiclass_f1_score



/home/chiara/anaconda3/envs/MGS/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-06-19 16:39:40.732552: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-19 16:39:41.669332: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Crea un oggetto ConfigParser
config = configparser.ConfigParser()

# Leggi il file di configurazione
config.read('/home/chiara/DataAUG/Classification/con.ini')

train_folder = config['AAA']['train_folder']
test_folder = config['AAA']['test_folder']
train_folder_GAN = config['AAA']['train_folder_GAN']
out_ss=len(os.listdir(train_folder))
in_ch=int(config['AAA']['in_ch'])
N_EPOCHS = int(config['AAA']['num_epoch'])
N = int(config['AAA']['N'])
BATCH_SIZE = int(config['AAA']['BATCH_SIZE'])
ss = int(config['AAA']['SS'])

drop = float(config['AAA']['DROP'])
lr = float(config['AAA']['LR'])

print(f'   \n input channel: {in_ch} \n output_channel: {out_ss}')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(0)
torch.manual_seed(0)

#Data
label_mapping = {
    'Backdoor_attack': 0,
    'DDoS_HTTP_Flood_Attacks': 1,
    'NET': 2,
    'Port_Scanning_attack': 3,
    'Ransomware_attack': 4,#-- 
    'SQL_injection_attack': 5,
    'Uploading_attack': 6,#--
    'Vulnerability_scanner_attack': 7
}

transform =  transforms.Compose(
            [ transforms.ToTensor(),transforms.Grayscale(num_output_channels=1), transforms.Normalize((0.5), (1))])
test = BalanceSet(root_path=test_folder, transform=transform, N = 400)

test_loader = DataLoader(test, batch_size=BATCH_SIZE, num_workers=24)



   
 input channel: 1 
 output_channel: 8


## TESTing

Model

In [3]:
path = '/home/chiara/DataAUG/Classification/MODELLI_finalLIIIIIIIII_'
#path = '/home/chiara/DataAUG/Classification/MODELLI_final_0'
Models = [i for i in os.listdir(path) if i.endswith('.pth')]

Models

['16_with_dropout_30DIFF.pth',
 '16_with_dropout_50DIFF.pth',
 '16_with_dropout_80DIFF.pth',
 '16_with_dropout_20DIFF.pth']

In [4]:
for i in range(0,len(Models)):
    Mod = Models[i]
    path_tmp = os.path.join(path,Mod)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    tmp_model = Network(1, out_ss, ss =ss, p = drop, up =0 ).to(device)

    tmp_model.load_state_dict(torch.load(path_tmp,  map_location=device))
    print(f'Model loaded {Mod}')
    tmp_model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Forward pass
            outputs = tmp_model(inputs, 0)
            # Thresholding
            predicted_classes = torch.argmax(outputs, dim=1)

            # Collect tensors arrays
            y_true.extend(targets.cpu().numpy())
            y_pred.extend(predicted_classes.cpu().numpy())

    # Calculate confusion matrix
    labels = list(label_mapping.values())
    #conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
    #print(conf_matrix)
    #print(classification_report(y_true, y_pred, labels=labels))
    print(f'F1: {multiclass_f1_score( torch.tensor(y_pred), torch.tensor(y_true), num_classes=8 )}')
    a=classification_report(y_true, y_pred, labels=labels, output_dict=True)
    f1_scores = [a[f'{i}']['f1-score'] for i in range(0, 7)]
    print(f'VAR f1 : {np.var(f1_scores)}')
    #print(f'Accuracy {a['accuracy']}')

Model loaded 16_with_dropout_30DIFF.pth
F1: 0.8574821352958679
VAR f1 : 0.021737380099219546
Model loaded 16_with_dropout_50DIFF.pth
F1: 0.8316932320594788
VAR f1 : 0.03597421788506873
Model loaded 16_with_dropout_80DIFF.pth
F1: 0.8028503060340881
VAR f1 : 0.037070775023822264
Model loaded 16_with_dropout_20DIFF.pth
F1: 0.8523922562599182
VAR f1 : 0.02135418470600151


In [19]:
a

{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0},
 '1': {'precision': 0.99,
  'recall': 0.9924812030075187,
  'f1-score': 0.9912390488110138,
  'support': 399.0},
 '2': {'precision': 1.0,
  'recall': 0.9950124688279302,
  'f1-score': 0.9975,
  'support': 401.0},
 '3': {'precision': 0.7361623616236163,
  'recall': 1.0,
  'f1-score': 0.8480340063761955,
  'support': 399.0},
 '4': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0},
 '5': {'precision': 0.9900497512437811,
  'recall': 0.9974937343358395,
  'f1-score': 0.9937578027465668,
  'support': 399.0},
 '6': {'precision': 0.9960629921259843,
  'recall': 0.6340852130325815,
  'f1-score': 0.774885145482389,
  'support': 399.0},
 '7': {'precision': 0.9899497487437185,
  'recall': 0.9874686716791979,
  'f1-score': 0.9887076537013801,
  'support': 399.0},
 'micro avg': {'precision': 0.9344741235392321,
  'recall': 0.9344741235392321,
  'f1-score': 0.9344741235392321,
  'support': 2396.0},
 'macro avg